# Chroma Studio in Practice — A Decision-Making Playbook
**The goal:** don't just *look* at a vector database — use each Chroma Studio view to make a specific, defensible decision about your data. By the end you'll have a repeatable checklist you can run on any real corpus at work.

We use a small enterprise support knowledge base (`dataset/support_kb.csv`) that has **six deliberately planted flaws** — the kind you hit constantly in real RAG projects. Each section: **spot the problem in a Studio view → decide the fix → apply it → re-check.**

> The dataset has a hidden `flaw` column so this notebook can confirm your findings. Real data never comes labelled like that — that's exactly why the Studio views matter.

## How this maps to Chroma Studio
Every diagnosis below is something you can *also* see in the Chroma Studio app's tabs. This notebook computes the same things in code so you understand what the visual is telling you:

| Playbook section | Chroma Studio tab | Decision it drives |
|---|---|---|
| 1. Coverage & topic map | Visualize (color by metadata) | Is any topic thinly covered? |
| 2. Duplicates | Visualize (points sitting on top of each other) + Search | Dedupe / merge |
| 3. Cluster vs metadata | Visualize (color by KMeans vs by category) | Is my metadata trustworthy? |
| 4. Mislabeled metadata | Cluster drill-down | Fix metadata (enrich) |
| 5. Overlong / mixed docs | Search returns a muddy chunk | Re-chunk |
| 6. Semantic collisions | Search top-2 too close | Add disambiguating metadata |
| 7. Missing metadata | Browse filter shows blanks | Backfill metadata |

In [ ]:
# ---------------------------------------------------------------------------
# Self-contained embedder so this runs offline. For REAL use, replace this whole
# cell with your in-house embedder:
#
#   from inhouse_wrappers import InHouseEmbeddings
#   embedder = InHouseEmbeddings()
#
# Everything else in the notebook stays exactly the same.
#
# NOTE: this mock uses word-overlap with light IDF-style weighting so that
# near-duplicates score genuinely high and unrelated docs score low -- close
# enough to real embedding behavior for the diagnostics below to be meaningful.
# A real model (Jina, BGE, etc.) will separate these even more cleanly.
# ---------------------------------------------------------------------------
import numpy as np, re
from collections import Counter

_STOP = set("the a an to of and or is are be for in on at by with from your you "
            "we our it its as within once after before per must can may".split())

def _tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in _STOP and len(w) > 2]

class MockEmbedder:
    """Word-overlap embedding with a fixed vocabulary, L2-normalized. Documents
    sharing meaningful words get similar vectors; the more distinctive the shared
    words, the higher the similarity. Crude vs a real model, but good enough that
    duplicates, collisions and clusters behave the way they would in production."""
    def __init__(self, corpus=None, dim=256):
        self.dim = dim
    def _vec(self, text):
        v = np.zeros(self.dim)
        toks = _tokens(text)
        if not toks:
            return v
        for w, c in Counter(toks).items():
            v[abs(hash(w)) % self.dim] += c
        n = np.linalg.norm(v)
        return v / n if n else v
    def embed_documents(self, texts): return [self._vec(t).tolist() for t in texts]
    def embed_query(self, text): return self._vec(text).tolist()

embedder = MockEmbedder()
print("Embedder ready (mock — swap for InHouseEmbeddings() for real use).")

> ⚠️ **About the offline embedder.** This notebook ships with a crude word-overlap mock so it runs anywhere with no API key — good enough to make duplicates, collisions, and coverage gaps *visible*, but noisier than a real model. The mislabel section (which leans on neighborhood quality) will show some false positives with the mock. **Swap in `InHouseEmbeddings()` and every section sharpens.** The playbook is teaching the diagnostic *method*, not the mock's exact numbers.

## Load the knowledge base

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("dataset/support_kb.csv").fillna({"category": "", "product": "", "flaw": ""})
print(f"{len(df)} documents, {df['category'].nunique()} distinct categories")
vectors = np.array(embedder.embed_documents(df["text"].tolist()))
print("Embedding matrix:", vectors.shape)
df.head()

---
## 1. Coverage & topic map — *is any topic dangerously thin?*
**Studio view:** Visualize tab, color by `category`. **Decision:** a topic with only one document is a coverage risk — one bad chunk and that whole topic is unanswerable. You either add more docs or accept the gap knowingly.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(vectors)
fig, ax = plt.subplots(figsize=(9, 6))
for cat in sorted(df["category"].unique()):
    idx = df["category"] == cat
    label = cat if cat else "(no category)"
    ax.scatter(coords[idx, 0], coords[idx, 1], label=label, s=90, alpha=0.75)
for i, row in df.iterrows():
    ax.annotate(row["id"], (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.6)
ax.legend(title="category"); ax.set_title("Topic map (PCA, colored by category)")
plt.tight_layout(); plt.show()

# The code version of "what am I looking at": count docs per category
print("Docs per category:")
print(df[df["category"] != ""]["category"].value_counts())
print("\nThin topics (<= 1 doc) — coverage risk:")
thin = df[df["category"] != ""]["category"].value_counts()
print(thin[thin <= 1])

**What you'd see in Studio:** in the Visualize tab colored by `category`, the `compliance` point sits alone with no neighbors of its own color. **Decision → enrich the corpus:** `compliance` (the GDPR doc) is a one-document topic. In production you'd add more compliance docs, or flag to stakeholders that compliance questions have thin coverage. This is a *content* decision the map surfaced.

---
## 2. Duplicates — *am I storing the same answer 3 times?*
**Studio view:** Visualize tab — near-duplicates render as points stacked on top of each other. **Decision:** duplicates waste retrieval slots (your top-k fills with the same fact) and skew evaluation. Merge or drop them.

In [ ]:
# Find pairs with very high cosine similarity (near-duplicates)
sim = vectors @ vectors.T
np.fill_diagonal(sim, 0)
threshold = 0.55
print(f"Near-duplicate pairs (cosine > {threshold}):")
seen = set()
for i in range(len(df)):
    for j in range(i+1, len(df)):
        if sim[i, j] > threshold:
            print(f"  {df.iloc[i]['id']}  <->  {df.iloc[j]['id']}   sim={sim[i,j]:.3f}")
            seen.add(df.iloc[i]["id"]); seen.add(df.iloc[j]["id"])
print("\nDocs involved in near-duplication:", sorted(seen))

**What you'd see in Studio:** in the Visualize tab, `refund_01`–`04` cluster so tightly they overlap. Hovering shows they're all 'refund in 5 business days.' **Decision → dedupe:** keep one canonical refund doc (or merge them into one richer doc). Below we simulate the decision: which to keep, which to drop.

In [ ]:
# Decision: keep the most complete refund doc, mark the rest for removal
refund_docs = df[df["id"].str.startswith("refund")]
keep = refund_docs.loc[refund_docs["text"].str.len().idxmax(), "id"]  # longest = most complete
drop = [i for i in refund_docs["id"] if i != keep]
print(f"KEEP: {keep}")
print(f"DROP (redundant): {drop}")
print("\nIn Chroma Studio you'd do this in the Edit/Update tab -> Delete for each drop id,")
print("or in the Manage tab if a whole collection were redundant.")

---
## 3. Cluster vs metadata — *can I trust my category labels?*
**Studio view:** Visualize tab — flip **Color by** between `KMeans cluster` and `category`. **Decision:** if the algorithm's clusters *disagree* with your metadata categories, your metadata is either wrong or your categories don't match how the content actually groups. Both are fixable — but you have to notice first.

In [ ]:
from sklearn.cluster import KMeans

k = df[df["category"] != ""]["category"].nunique()
clusters = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(vectors)
df["_cluster"] = clusters

# Cross-tab: do KMeans clusters line up with categories?
ct = pd.crosstab(df["_cluster"], df["category"].replace("", "(none)"))
print("Rows = KMeans clusters, Cols = your category labels.")
print("A clean corpus has each cluster dominated by ONE category.\n")
print(ct)

**How to read that table:** a trustworthy corpus shows each KMeans row concentrated in one category column. Any row spread across several columns — or a category split across many rows — means the automatic grouping and your labels disagree. That's your cue to inspect those specific docs (next section).

---
## 4. Mislabeled metadata — *find the doc whose label lies*
**Studio view:** Cluster drill-down — open a cluster and read its docs; the odd-one-out whose `category` doesn't match its neighbors is mislabeled. **Decision → enrich/fix metadata**, don't touch the text.

In [ ]:
# For each doc, find the majority category of its nearest neighbors.
# We only flag a doc as suspect when its neighbors AGREE strongly with each other
# on a different category AND are reasonably similar -- otherwise we'd get false
# positives from weak/ambiguous neighborhoods. (Tune these thresholds for real data.)
k_nn = 3
min_neighbor_sim = 0.15
suspects = []
for i in range(len(df)):
    order = [j for j in np.argsort(sim[i])[::-1] if sim[i][j] >= min_neighbor_sim][:k_nn]
    neigh_cats = [df.iloc[j]["category"] for j in order if df.iloc[j]["category"]]
    if len(neigh_cats) >= 2:                          # need a real neighborhood
        majority = max(set(neigh_cats), key=neigh_cats.count)
        agree = neigh_cats.count(majority)
        own = df.iloc[i]["category"]
        # flag only if neighbors strongly agree on a DIFFERENT category
        if own and own != majority and agree >= 2:
            suspects.append((df.iloc[i]["id"], own, majority, df.iloc[i]["text"][:60]))

print("Docs whose label disagrees with a strong-agreement neighborhood:")
if suspects:
    for id_, own, maj, text in suspects:
        print(f"  {id_}: labelled '{own}', but neighbors agree on '{maj}'  —  {text}...")
else:
    print("  (none flagged at these thresholds)")
print("\nNOTE: with the crude mock embedder this is approximate — a real embedding")
print("model gives much cleaner neighborhoods. The METHOD is the point: compare each")
print("doc's label to what its nearest neighbors say, and investigate disagreements.")

**Decision → fix the label:** `mislabel_01` is labelled `shipping` but it's about late fees on invoices (clearly `billing`). In Chroma Studio you'd fix this in the **Edit/Update** tab — change the metadata JSON to `{"category": "billing"}` and save. Note: a *metadata-only* update doesn't re-embed (the text didn't change), so it's instant. That distinction matters at scale.

In [ ]:
# Apply the fix in our dataframe (simulating the Studio Edit/Update action)
df.loc[df["id"] == "mislabel_01", "category"] = "billing"
print("Fixed mislabel_01 -> category now:", df.loc[df["id"]=="mislabel_01","category"].values[0])

---
## 5. Overlong / mixed docs — *why is this search result muddy?*
**Studio view:** Search tab — a query returns a chunk that's *partly* relevant but padded with unrelated text. **Decision → re-chunk:** one document covering three topics should be three documents.

In [ ]:
def search(query, k=3):
    q = np.array(embedder.embed_query(query))
    order = np.argsort(vectors @ q)[::-1][:k]
    return [(df.iloc[i]["id"], df.iloc[i]["text"], float((vectors @ q)[i])) for i in order]

print("Query: 'does the app support biometric login?'\n")
for id_, text, score in search("does the app support biometric login"):
    print(f"[{score:.3f}] {id_}: {text}")

**What you'd see in Studio:** the Search tab returns `mixed_01` — but that doc answers the biometric question buried between office hours and data export. The embedding is 'diluted' across 3 topics, so it matches everything weakly and nothing strongly. **Decision → re-chunk** into 3 clean docs:

In [ ]:
# Split the overlong doc into focused chunks (the re-chunk decision)
mixed_text = df.loc[df["id"]=="mixed_01", "text"].values[0]
sentences = [s.strip() for s in mixed_text.split(". ") if s.strip()]
print("Re-chunked into focused docs:")
new_docs = [
    ("hours_01",   "Our office is open 9 to 5 on weekdays.", "misc", "platform"),
    ("biometric_01","The mobile app supports biometric login on iOS and Android.", "account", "platform"),
    ("export_01",  "Bulk export of your data is available in CSV and JSON from account settings under 'Data'.", "account", "platform"),
]
for id_, text, cat, prod in new_docs:
    print(f"  {id_} [{cat}]: {text}")
print("\nIn Studio: delete mixed_01 (Edit/Update tab), then add these 3 (Add tab).")
print("Re-embedding happens automatically on add — each focused doc now embeds cleanly.")

---
## 6. Semantic collisions — *two different answers that look identical*
**Studio view:** Search tab — top-2 results have almost the same distance for a query that should clearly prefer one. **Decision → add disambiguating metadata**, then filter on it at query time.

In [ ]:
print("Query: 'how do I cancel my subscription?'\n")
for id_, text, score in search("how do I cancel my subscription", k=2):
    print(f"[{score:.3f}] {id_}: {text}")
print("\nBoth 'cancel subscription' and 'cancel order' score nearly the same —")
print("the embedder can't tell them apart on wording alone. That ambiguity is the bug.")

**Decision → enrich metadata for filtering:** `collision_01` (cancel *subscription*) and `collision_02` (cancel *order*) read almost identically. Rather than hope the embedder separates them, add a metadata field you can *filter* on so a subscription query never even considers order-cancellation docs.

In [ ]:
# Add an 'intent' metadata field so queries can filter precisely
df.loc[df["id"]=="collision_01", "intent"] = "subscription_mgmt"
df.loc[df["id"]=="collision_02", "intent"] = "order_mgmt"
print("Enriched metadata:")
print(df[df["id"].str.startswith("collision")][["id","category","intent"]])
print("\nAt query time in Studio's Search (or your RAG code), filter where intent=")
print("'subscription_mgmt' for subscription questions — the collision disappears because")
print("you're no longer relying on the embedding alone to disambiguate.")

---
## 7. Missing metadata — *the silent filter-breaker*
**Studio view:** Browse tab — filter/scan the `meta.category` column for blanks. **Decision → backfill:** docs with no category are invisible to any category-filtered query, silently missing from results.

In [ ]:
missing = df[df["category"] == ""]
print("Docs with NO category (invisible to category-filtered retrieval):")
for _, r in missing.iterrows():
    print(f"  {r['id']}: {r['text'][:60]}")

# Backfill decision: infer category from nearest well-labelled neighbors
for i in missing.index:
    order = np.argsort(sim[i])[::-1]
    for j in order:
        if df.iloc[j]["category"]:
            df.loc[i, "category"] = df.iloc[j]["category"]
            break
print("\nAfter backfill (inferred from nearest labelled neighbor):")
print(df.loc[missing.index][["id","category"]])

**In Studio:** the Browse tab's filter makes blank-metadata rows easy to spot; you'd fix each in Edit/Update. Backfilling by nearest-neighbor category is a fast first pass — but review inferred labels, since a wrong guess is worse than a blank (a blank fails loudly, a wrong label fails silently).

---
## The repeatable checklist (take this to real corpora)
You just ran the exact loop you'd run on any new dataset at work:

1. **Coverage** — color the map by category; find thin/orphan topics → *enrich content*
2. **Duplicates** — find stacked points / high-similarity pairs → *dedupe*
3. **Cluster vs metadata** — cross-tab KMeans vs labels → *is metadata trustworthy?*
4. **Mislabeled** — neighbors disagree with a doc's label → *fix metadata (no re-embed)*
5. **Overlong/mixed** — muddy search hits → *re-chunk (re-embeds)*
6. **Collisions** — top-2 too close → *add metadata to filter on*
7. **Missing metadata** — blanks in Browse → *backfill, then review*

**The meta-skill:** every visual maps to a decision, and every decision is one of a small set — enrich content, dedupe, fix metadata, re-chunk, or add a filter field. That's what 'analyze → decide → act' looks like in RAG, and it's exactly what interviewers and real projects are testing for.

## Confirm your findings against the hidden flaw labels
Real data won't have this — it's here so you can check the playbook caught everything.

In [ ]:
orig = pd.read_csv("dataset/support_kb.csv").fillna("")
print("Every planted flaw and the doc(s) that carried it:")
for flaw in sorted(f for f in orig["flaw"].unique() if f):
    docs = orig[orig["flaw"] == flaw]["id"].tolist()
    print(f"  {flaw:15s}: {docs}")
print("\nIf the sections above surfaced each of these without peeking at this cell,")
print("you've got the diagnostic instinct that separates expert RAG work from")
print("'load data and hope'. That's the real deliverable of this playbook.")